# 다봐요 (Dabwayo) — GPU Generation Server (Google Colab)

Runs an **open-weight video model** on a Colab GPU and exposes it over a public
URL. Dabwayo's `remote` provider calls it, so `generate_video(..., provider=
'remote')` produces real footage that composites into your timeline.

### What fits which runtime (read this)
A **free T4** has ~16 GB VRAM but only ~**12.7 GB system RAM** and **no
bfloat16**. That RAM is the wall: big models (LTX-Video pulls a ~9 GB T5-XXL
text encoder) OOM **while loading**, which **kills the kernel** — uncatchable,
so the session just dies. Therefore:

| `MODEL` | Runtime | Modes | Notes |
|---------|---------|-------|-------|
| **`light`** (default) | **free T4** | t2v | Zeroscope — light, reliable on free Colab |
| `ltx` | Colab **Pro / High-RAM** | t2v + i2v | Photoreal; needs more RAM than the free tier |

For **photoreal i2v on the free tier**, the practical route is a paid API key
(`FAL_KEY` / `REPLICATE_API_TOKEN`) — already supported by Dabwayo's `fal` /
`replicate` providers, no Colab needed.

**Steps:** Runtime → Change runtime type → **GPU**, then Run all. Copy the
printed `DABWAYO_VIDEOGEN_URL`:
```bash
export DABWAYO_VIDEOGEN_URL='https://xxxx.trycloudflare.com'
export DABWAYO_VIDEOGEN_PROVIDER=remote
```
Keep this tab running — closing it stops the server.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU! Runtime -> Change runtime type -> GPU'

## 2. Install dependencies

In [ ]:
%pip -q install "diffusers>=0.32" "transformers>=4.44" accelerate safetensors \
    imageio imageio-ffmpeg fastapi "uvicorn[standard]" nest-asyncio pillow
print('deps installed')

## 3. Load the model
**Default `MODEL='light'` (Zeroscope, t2v)** loads reliably on a free T4.
Switch to `MODEL='ltx'` **only on Colab Pro / High-RAM** for photoreal t2v+i2v
— on the free tier it OOMs *during load* and kills the kernel.

In [ ]:
import torch, gc
MODEL = 'light'   # 'light' (free T4, t2v) | 'ltx' (Pro/High-RAM, t2v+i2v)
DTYPE = torch.float16   # T4 has NO bfloat16
BACKEND = None; t2v = i2v = None
def _free(): gc.collect(); torch.cuda.empty_cache()

def load_light():
    global t2v, i2v, BACKEND
    from diffusers import DiffusionPipeline
    t2v = DiffusionPipeline.from_pretrained('cerspense/zeroscope_v2_576w',
                                            torch_dtype=DTYPE)
    t2v.enable_model_cpu_offload()
    try: t2v.enable_vae_slicing()
    except Exception: pass
    i2v = None; BACKEND = 'light'

def load_ltx():
    global t2v, i2v, BACKEND
    from diffusers import LTXPipeline, LTXImageToVideoPipeline
    from transformers import T5EncoderModel, BitsAndBytesConfig
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','-q','install','bitsandbytes','sentencepiece'])
    mid = 'Lightricks/LTX-Video'
    te = T5EncoderModel.from_pretrained(mid, subfolder='text_encoder',
        quantization_config=BitsAndBytesConfig(load_in_8bit=True), torch_dtype=DTYPE)
    t2v = LTXPipeline.from_pretrained(mid, text_encoder=te, torch_dtype=DTYPE)
    t2v.enable_sequential_cpu_offload()
    try: t2v.vae.enable_tiling(); t2v.vae.enable_slicing()
    except Exception: pass
    i2v = LTXImageToVideoPipeline(**t2v.components); BACKEND = 'ltx'

# Note: a system-RAM OOM kills the kernel and CANNOT be caught here; that is
# why 'light' is the default. We only auto-fall-back on catchable errors.
try:
    (load_ltx if MODEL == 'ltx' else load_light)()
except Exception as e:
    print('load failed -> light:', repr(e)[:200]); t2v=i2v=None; _free(); load_light()
print('READY backend =', BACKEND, '| modes =', ['t2v'] + (['i2v'] if i2v else []))

## 4. Generation API (`GET /health`, `POST /generate` -> video/mp4)

In [ ]:
import base64, io, tempfile
from PIL import Image
from fastapi import FastAPI, Request, Response, HTTPException
from diffusers.utils import export_to_video
def _snap(v,m,lo): return max(lo, int(round(v/m))*m)

def run_generation(body):
    prompt = body.get('prompt',''); mode = body.get('mode','t2v')
    fps = float(body.get('fps',24)); steps = int(body.get('steps',25))
    seed = body.get('seed')
    gen = torch.Generator(device='cuda').manual_seed(int(seed)) if seed is not None else None
    if BACKEND == 'ltx':
        w=_snap(int(body.get('width',704)),32,32); h=_snap(int(body.get('height',480)),32,32)
        nf=max(9,int(round((int(body.get('num_frames',73))-1)/8))*8+1)
        kw=dict(prompt=prompt,width=w,height=h,num_frames=nf,num_inference_steps=steps,
                guidance_scale=float(body.get('guidance',3.0)),generator=gen)
        if mode=='i2v':
            if not body.get('image_b64'): raise HTTPException(400,'i2v needs image_b64')
            img=Image.open(io.BytesIO(base64.b64decode(body['image_b64']))).convert('RGB')
            frames=i2v(image=img.resize((w,h)),**kw).frames[0]
        else: frames=t2v(**kw).frames[0]
    else:
        if mode=='i2v': raise HTTPException(400, "i2v needs MODEL='ltx' (Colab Pro)")
        w=_snap(int(body.get('width',576)),8,256); h=_snap(int(body.get('height',320)),8,256)
        nf=int(body.get('num_frames',24))
        frames=t2v(prompt,num_frames=nf,height=h,width=w,num_inference_steps=steps,generator=gen).frames[0]
    path=tempfile.mktemp(suffix='.mp4'); export_to_video(frames,path,fps=fps); _free()
    return open(path,'rb').read()

app=FastAPI()
@app.get('/health')
def health(): return {'ok':True,'backend':BACKEND,'modes':['t2v']+(['i2v'] if i2v else []),
    'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
@app.post('/generate')
async def generate(request: Request):
    body=await request.json()
    try: return Response(content=run_generation(body), media_type='video/mp4')
    except torch.cuda.OutOfMemoryError:
        _free(); raise HTTPException(507,'GPU OOM - lower num_frames/width/height')
print('API defined')

## 5. Public tunnel + launch

In [ ]:
import nest_asyncio, threading, uvicorn, subprocess, re, time, os, urllib.request, stat
nest_asyncio.apply()
def _serve(): uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=_serve, daemon=True).start(); time.sleep(3)
BIN='/usr/local/bin/cloudflared'
if not os.path.exists(BIN):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)
proc=subprocess.Popen([BIN,'tunnel','--url','http://localhost:8000','--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for line in proc.stdout:
    m=re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n'+'='*60); print('DABWAYO_VIDEOGEN_URL =', url); print('='*60)
print(f"export DABWAYO_VIDEOGEN_URL='{url}'"); print('export DABWAYO_VIDEOGEN_PROVIDER=remote')
print('Keep this tab open to keep the server alive.')

## 6. (Optional) Smoke test

In [ ]:
import requests
print('health:', requests.get(url+'/health', timeout=30).json())
r=requests.post(url+'/generate', json={'prompt':'a golden retriever running on a beach, slow motion',
    'mode':'t2v','num_frames':24,'fps':24,'steps':25}, timeout=1200)
open('test.mp4','wb').write(r.content); print('wrote test.mp4', len(r.content),'bytes')
from IPython.display import Video; Video('test.mp4', embed=True)